# Assess dataset coverage

Detect class imbalance, metadata gaps, and embedding blind spots using
the config-driven `data-coverage` workflow.

## What you'll do

- Load MNIST from HuggingFace and build a **deliberately biased** subset
  that simulates a flawed data collection pipeline
- Attach synthetic metadata factors (lighting condition, collection
  facility) with intentional gaps for certain classes
- Run the `data-coverage` workflow **without** an extractor to surface
  label and metadata issues quickly
- Re-run **with** a BoVW extractor to add embedding coverage and
  dimensional completeness analysis
- Read the built-in **coverage report** and drill into raw results
- Tune **health thresholds** to control when findings become warnings

## What you'll learn

- How uneven data collection creates coverage gaps that are invisible
  until you explicitly measure them
- How to configure and run the `data-coverage` workflow via `run_task()`
- How to interpret the five coverage dimensions: embedding coverage,
  dimensional completeness, label distribution, metadata distribution,
  and metadata gaps
- The difference between running with and without an extractor
- How to adjust health thresholds for different risk tolerances

## What you'll need

- `dataeval-flow` (includes `dataeval`, `datasets`, `maite-datasets`, `pydantic`)
- Internet connection (to download MNIST from HuggingFace Hub on first run)

### Step-by-step guide

## Data Preparation: Build a biased dataset

Imagine you're building a digit recognition model for a postal sorting
system. Your data comes from three sorting facilities:

- **Facility A** — handles digits 0–3, well-staffed, collected plenty
  of images under both bright and dim lighting
- **Facility B** — handles digits 4–6, moderate collection effort
- **Facility C** — handles digits 7–9, understaffed, collected far
  fewer images and almost exclusively under bright lighting

This kind of uneven collection is common in real-world ML pipelines.
The result is a dataset with **class imbalance** (fewer samples of
7–9), **metadata gaps** (dim lighting nearly absent for 7–9), and
**poor embedding coverage** in the under-represented region.

We'll simulate this by subsampling MNIST and attaching synthetic
metadata.

In [ ]:
from typing import Any, cast

import numpy as np
from datasets import Dataset
from datasets import load_dataset as hf_load
from maite_datasets.adapters import from_huggingface

# Download MNIST test split (10 000 images)
mnist_test = cast(Dataset, hf_load("ylecun/mnist", split="test"))
mnist_maite = from_huggingface(mnist_test)

print(f"Loaded {len(mnist_maite)} MNIST test images")

### Subsample with collection bias

We take different numbers of samples per digit class to mimic the
uneven collection across the three facilities.

In [ ]:
from collections.abc import Mapping

from dataeval.protocols import DatasetMetadata
from numpy.typing import NDArray

rng = np.random.default_rng(42)

# Per-class sample budgets — Facility C (digits 7–9) gets far fewer
samples_per_class = {
    0: 200, 1: 200, 2: 200, 3: 200,  # Facility A — well-covered
    4: 120, 5: 120, 6: 120,           # Facility B — moderate
    7: 30,  8: 30,  9: 30,            # Facility C — under-represented
}

# Collect indices per class, then subsample
selected_indices: list[int] = []
labels_full = np.array([int(np.asarray(mnist_maite[i][1]).argmax()) for i in range(len(mnist_maite))])

for cls, n in samples_per_class.items():
    cls_indices = np.where(labels_full == cls)[0]
    chosen = rng.choice(cls_indices, size=min(n, len(cls_indices)), replace=False)
    selected_indices.extend(chosen.tolist())

selected_indices.sort()
print(f"Selected {len(selected_indices)} images with biased class distribution")
print(f"Per-class counts: { {cls: n for cls, n in samples_per_class.items()} }")

### Attach synthetic metadata

We add two metadata factors that simulate real collection conditions:

- **lighting** — `"bright"` or `"dim"`. Facility C rarely collected
  dim lighting samples, creating a metadata gap for digits 7–9.
- **facility** — `"A"`, `"B"`, or `"C"`. Encodes where the image was
  collected. Strongly correlated with class, which gap analysis will
  detect.

In [ ]:
class BiasedMNIST:
    """Wraps a MAITE dataset with subsampling and synthetic metadata factors.

    Simulates a postal sorting system with uneven data collection across
    three facilities, each handling different digit classes under different
    lighting conditions.
    """

    def __init__(
        self,
        dataset: Any,
        indices: list[int],
        rng: np.random.Generator,
    ) -> None:
        self._dataset = dataset
        self._indices = indices

        # Pre-compute labels for metadata assignment
        self._labels = np.array(
            [int(np.asarray(dataset[i][1]).argmax()) for i in indices]
        )

        # Assign facility based on digit class
        self._facility = np.array([
            "A" if lbl <= 3 else "B" if lbl <= 6 else "C"
            for lbl in self._labels
        ])

        # Assign lighting — dim is common for Facility A/B, rare for C
        self._lighting = np.empty(len(indices), dtype=object)
        for i, lbl in enumerate(self._labels):
            if lbl <= 6:
                # Facility A/B: 40% dim, 60% bright
                self._lighting[i] = rng.choice(["bright", "dim"], p=[0.6, 0.4])
            else:
                # Facility C: 95% bright, 5% dim — a metadata gap
                self._lighting[i] = rng.choice(["bright", "dim"], p=[0.95, 0.05])

    def __len__(self) -> int:
        return len(self._indices)

    @property
    def metadata(self) -> DatasetMetadata:
        return DatasetMetadata(
            id="biased-mnist",
            index2label={i: str(i) for i in range(10)},
        )

    def __getitem__(self, index: int) -> tuple[NDArray[Any], Any, Mapping[str, Any]]:
        orig_idx = self._indices[index]
        image, target, _ = self._dataset[orig_idx]

        metadata: dict[str, Any] = {
            "lighting": self._lighting[index],
            "facility": self._facility[index],
        }

        return np.asarray(image), target, metadata


biased_dataset = BiasedMNIST(mnist_maite, selected_indices, rng)
print(f"BiasedMNIST: {len(biased_dataset)} samples")

# Spot-check metadata
for i in [0, 100, len(biased_dataset) - 1]:
    _, target, meta = biased_dataset[i]
    label = int(np.asarray(target).argmax())
    print(f"  Sample {i}: digit={label}, lighting={meta['lighting']}, facility={meta['facility']}")

### Visualize the collection bias

Before running the workflow, let's see what the imbalance looks like.

In [ ]:
import matplotlib.pyplot as plt

class_counts = {cls: samples_per_class[cls] for cls in range(10)}
colors = ["#2ecc71"] * 4 + ["#f39c12"] * 3 + ["#e74c3c"] * 3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution
ax1.bar([str(c) for c in class_counts], class_counts.values(), color=colors)
ax1.set_xlabel("Digit class")
ax1.set_ylabel("Sample count")
ax1.set_title("Class distribution (biased collection)")
ax1.axhline(y=np.mean(list(class_counts.values())), color="gray", linestyle="--", label="mean")
ax1.legend()

# Lighting distribution per facility
facilities = ["A", "B", "C"]
bright_counts = []
dim_counts = []
for fac in facilities:
    mask = biased_dataset._facility == fac
    bright_counts.append(int(np.sum(biased_dataset._lighting[mask] == "bright")))
    dim_counts.append(int(np.sum(biased_dataset._lighting[mask] == "dim")))

x = np.arange(len(facilities))
ax2.bar(x - 0.15, bright_counts, 0.3, label="bright", color="#f1c40f")
ax2.bar(x + 0.15, dim_counts, 0.3, label="dim", color="#34495e")
ax2.set_xticks(x)
ax2.set_xticklabels(facilities)
ax2.set_xlabel("Facility")
ax2.set_ylabel("Sample count")
ax2.set_title("Lighting distribution per facility")
ax2.legend()

plt.tight_layout()
plt.show()

The charts show exactly the kind of blind spots we want to catch:
- Facility C (digits 7–9) has far fewer samples overall
- Facility C has almost no dim lighting samples

A model trained on this data might perform well on bright images of
common digits but fail on dim images of digits 7, 8, or 9.

## Step 1: Run coverage without an extractor (metadata only)

The `data-coverage` workflow can run **without** an extractor for a
fast first pass. This skips embedding coverage and completeness but
still analyzes label distribution, metadata distribution, and metadata
gaps — often the most actionable findings.

In [ ]:
from pathlib import Path

from dataeval_flow.config import PipelineConfig, SourceConfig
from dataeval_flow.config.schemas import (
    DataCoverageTaskConfig,
    DataCoverageWorkflowConfig,
    DatasetProtocolConfig,
)
from dataeval_flow.workflow import run_task
from dataeval_flow.workflows.coverage.params import DataCoverageHealthThresholds

metadata_only_workflow = DataCoverageWorkflowConfig(
    name="coverage-metadata-only",
    run_gap_analysis=True,
    gap_mi_threshold=0.1,        # Include factors with MI ≥ 0.1
    gap_min_representation=5,    # Flag class-factor-value combos with < 5 samples
    balance=True,
    diversity_method="simpson",
    health_thresholds=DataCoverageHealthThresholds(
        class_imbalance_ratio=3.0,  # Tight — we want to catch moderate imbalance
        gap_count=2,                # Warn if ≥ 2 gaps found
    ),
)

task_metadata = DataCoverageTaskConfig(
    name="postal-coverage-metadata",
    workflow="coverage-metadata-only",
    sources="postal_src",
    # No extractor — metadata-only pass
)

config_metadata = PipelineConfig(
    datasets=[
        DatasetProtocolConfig(name="postal_biased", format="maite", dataset=biased_dataset),
    ],
    sources=[
        SourceConfig(name="postal_src", dataset="postal_biased"),
    ],
    workflows=[metadata_only_workflow],
    tasks=[task_metadata],
)

In [ ]:
result_metadata = run_task(task_metadata, config_metadata, cache_dir=Path("./cache"))

### Coverage report (metadata only)

The report summarizes label distribution, metadata distribution, and
metadata gaps. Since we didn't configure an extractor, embedding
coverage and completeness are skipped.

In [ ]:
print(result_metadata.report())

### Interpreting the findings

The report should reveal several issues:

- **Label Distribution** — The 200:30 ratio between the largest and
  smallest classes (≈6.7:1) exceeds our 3:1 threshold, triggering a
  warning. Digits 7, 8, 9 are severely under-represented.
- **Metadata Coverage Gaps** — The gap analysis cross-references class
  labels with metadata factors. It should flag that digits 7–9 have
  almost no `dim` lighting samples — a blind spot that could hurt
  real-world performance.
- **Metadata Distribution** — Shows the `lighting` and `facility`
  factors and their value counts.

### Drill into raw results

The `result.data.raw` object provides machine-readable access to every
metric for programmatic inspection.

In [ ]:
raw = result_metadata.data.raw

# Label distribution
ld = raw.label_distribution
print(f"Number of classes: {ld.num_classes}")
print(f"Empty images: {len(ld.empty_images)}")
print("\nClass distribution:")
for cls, count in sorted(ld.class_distribution.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cls}: {count}")

In [ ]:
# Metadata gaps — the most actionable finding
if raw.metadata_gaps and raw.metadata_gaps.gaps:
    print("Metadata coverage gaps:")
    print(f"  Total gaps found: {len(raw.metadata_gaps.gaps)}\n")

    print("  Mutual information (class → factor):")
    for factor, mi in raw.metadata_gaps.mutual_info_class_to_factor.items():
        print(f"    {factor}: {mi:.4f}")

    print("\n  Gap details:")
    for gap in raw.metadata_gaps.gaps:
        print(
            f"    {gap.class_name} × {gap.factor_name}={gap.factor_value}: "
            f"count={gap.class_count}, expected={gap.expected_count:.1f}, "
            f"deficit={gap.deficit:.0%}"
        )
else:
    print("No metadata gaps detected.")

## Step 2: Run coverage with an extractor (full analysis)

Now we add a **BoVW** (Bag of Visual Words) extractor to unlock
embedding-based analyses: **embedding coverage** and **dimensional
completeness**. These tell us whether the dataset's feature space has
blind spots that metadata alone can't reveal.

Embedding coverage checks whether the feature space has uncovered
regions — areas where the model would encounter inputs unlike anything
in the training data. Dimensional completeness measures how
effectively the data explores the embedding dimensions.

In [ ]:
from dataeval.config import set_max_processes

from dataeval_flow.config import BoVWExtractorConfig

set_max_processes(8)

full_workflow = DataCoverageWorkflowConfig(
    name="coverage-full",
    coverage_method="adaptive",    # Adaptive coverage — recommended default
    coverage_percent=0.01,         # 1% of observations per coverage check
    num_observations=50,           # Number of neighbors for coverage analysis
    run_completeness=True,         # Measure dimensional completeness
    run_gap_analysis=True,
    balance=True,
    diversity_method="simpson",
    health_thresholds=DataCoverageHealthThresholds(
        uncovered_rate=10.0,        # Warn if > 10% uncovered in embedding space
        completeness_score=0.5,     # Warn if completeness < 0.5
        class_imbalance_ratio=3.0,  # Keep the tight threshold
        gap_count=2,
    ),
)

task_full = DataCoverageTaskConfig(
    name="postal-coverage-full",
    workflow="coverage-full",
    sources="postal_src",
    extractor="bovw_ext",
)

config_full = PipelineConfig(
    datasets=[
        DatasetProtocolConfig(name="postal_biased", format="maite", dataset=biased_dataset),
    ],
    sources=[
        SourceConfig(name="postal_src", dataset="postal_biased"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=512, batch_size=32),
    ],
    workflows=[full_workflow],
    tasks=[task_full],
)

In [ ]:
result_full = run_task(task_full, config_full, cache_dir=Path("./cache"))

### Full coverage report

Now the report includes embedding coverage and dimensional
completeness in addition to label and metadata findings.

In [ ]:
print(result_full.report())

### Understanding the new findings

With the extractor enabled, two new sections appear:

- **Embedding Coverage** — Reports how many observations are
  "uncovered" in embedding space. A high uncovered rate means there
  are regions of the feature space where the model has no training
  support. With our biased dataset, the under-represented classes
  (7–9) contribute to sparse regions that the coverage algorithm
  detects.

- **Dimensional Completeness** — A score between 0 and 1 measuring how
  well the data fills the embedding dimensions. Low completeness
  suggests the data clusters in a narrow subspace, leaving many
  directions unexplored. A biased dataset like ours tends to have
  lower completeness because the under-represented classes don't
  contribute enough variation.

### Inspect embedding results

In [ ]:
raw_full = result_full.data.raw

# Embedding coverage
if raw_full.coverage:
    cov = raw_full.coverage
    print("Embedding Coverage:")
    print(f"  Method: {cov.method}")
    print(f"  Uncovered: {cov.uncovered_count} ({cov.uncovered_rate:.1%})")
    print(f"  Radius: {cov.coverage_radius:.4f}")
else:
    print("No embedding coverage (extractor not configured)")

print()

# Dimensional completeness
if raw_full.completeness:
    comp = raw_full.completeness
    print("Dimensional Completeness:")
    print(f"  Score: {comp.completeness_score:.3f}")
    print(f"  Nearest neighbor pairs: {len(comp.nearest_neighbor_pairs)}")
else:
    print("No completeness data (extractor not configured)")

## Step 3: Tune health thresholds

Health thresholds control when findings escalate from `info` to
`warning`. The right thresholds depend on your domain:

| Metric | Default | Safety-critical | Web-scraped data |
|---|---|---|---|
| `uncovered_rate` | 10% | 3–5% | 15–20% |
| `completeness_score` | 0.5 | 0.7–0.8 | 0.3–0.4 |
| `class_imbalance_ratio` | 5:1 | 2–3:1 | 10–20:1 |
| `gap_count` | 3 | 1 | 5–10 |

For a safety-critical application like postal sorting where
misclassification has real cost, tighten the thresholds:

In [ ]:
strict_thresholds = DataCoverageHealthThresholds(
    uncovered_rate=5.0,          # Flag at 5% uncovered
    completeness_score=0.6,      # Require higher completeness
    class_imbalance_ratio=2.0,   # Very tight — almost balanced
    gap_count=1,                 # Any gap is a warning
)

strict_workflow = full_workflow.model_copy(
    update={"name": "coverage-strict", "health_thresholds": strict_thresholds},
)

task_strict = DataCoverageTaskConfig(
    name="postal-coverage-strict",
    workflow="coverage-strict",
    sources="postal_src",
    extractor="bovw_ext",
)

config_strict = PipelineConfig(
    datasets=config_full.datasets,
    sources=config_full.sources,
    extractors=config_full.extractors,
    workflows=[full_workflow, strict_workflow],
    tasks=[task_strict],
)

result_strict = run_task(task_strict, config_strict, cache_dir=Path("./cache"))
print(result_strict.report())

With stricter thresholds, more findings escalate to warnings — the
same data now triggers more alerts. This is the right behavior for
safety-critical deployments where you'd rather over-flag than miss
a coverage gap.

## Results Exploration: Export results

In [ ]:
json_str = result_full.export(fmt="json")
print(f"JSON output: {len(json_str)} characters")
print(json_str[:500] + "\n...")

## Conclusion

In this tutorial you learned how to:

- **Simulate a biased dataset** that mirrors real-world uneven data
  collection across facilities and conditions
- **Run coverage without an extractor** for a fast metadata-only pass
  that catches class imbalance and metadata gaps
- **Add an extractor** to unlock embedding coverage and dimensional
  completeness — revealing blind spots in feature space
- **Read the coverage report** — a single `result.report()` call for
  all five coverage dimensions with health status
- **Drill into raw results** for programmatic access to gap details,
  coverage metrics, and distribution data
- **Tune health thresholds** to match your domain's risk tolerance

The key takeaway: **low coverage is often invisible** until you
measure it. A dataset can have thousands of samples and still have
critical blind spots — missing conditions, under-represented classes,
or sparse regions in embedding space. Running the `data-coverage`
workflow before training helps you catch these issues early, when
they're cheapest to fix.

## What's next

- **Data cleaning** — Use the `data-cleaning` workflow to flag
  outliers and duplicates in your dataset before training
- **Drift monitoring** — After deploying, use `drift-monitoring` to
  detect when incoming data drifts away from your training distribution
- **Targeted collection** — Use the gap analysis results to guide
  additional data collection: specifically gather dim lighting samples
  from Facility C to fill the identified gaps
- **Run in Docker** — See the [Containerized Workflows how-to](../how_to/containerized_workflows.md) to
  build a container image, write a YAML config, and run a workflow with `docker run`